<a href="https://colab.research.google.com/github/pop123-ux/doc-assistant-hf/blob/main/coded.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook represents the pipeline of building an app where users upload PDFs, Word files, or text, and receive concise summaries or answers to specific questions about the document

- facebook/bart-large-cnn for summarization and deepset/roberta-base-squad2 for question answering

- Toggle to switch between abstractive (rewriting) and extractive (bullet points) summarization

In [1]:
!hf auth login

Hint: A new version of huggingface_hub (1.26.0) is available! You are using version 1.23.0.
To update, run: hf update
? How would you like to log in?  [Use arrows, Enter to confirm]
> Log in with your browser
  Paste an access token
? How would you like to log in? Log in with your browser

    Open this URL in your browser:
        https://hf.co/oauth/device

    And enter the code: WB93-GY94

    Waiting for authorization..............
Token is valid.
The token `oauth-pop123ux` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `oauth-pop123ux`
Note: This token will be refreshed automatically when it expires.


In [1]:
import json
import os
import torch
from datasets import Dataset, load_dataset
from huggingface_hub import hf_hub_download
from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

dataset1 = load_dataset('knkarthick/samsum')


In [ ]:
dataset1

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

In [3]:
model1 = "facebook/bart-large-cnn"

tokenizer1 = AutoTokenizer.from_pretrained(model1, use_fast=True)
tokenizer1.pad_token = tokenizer1.eos_token
tokenizer1.padding_side = "right"

In [4]:
# Now the tokenize function

def tokenize(examples):
  input_text = [f"{d}" for d in examples['dialogue']]

  model_inputs = tokenizer1(input_text, max_length=512, truncation=True, padding=False) # The data collator handles padding dynamically

  labels = tokenizer1(text_target=examples['summary'], max_length=128, truncation=True, padding=False)

  labels['input_ids'] = [
      [(token if token != tokenizer1.pad_token_id else -100) for token in label] for label in labels['input_ids']
  ]
  model_inputs['labels'] = labels['input_ids']

  return model_inputs

In [5]:
tokenized_dataset = dataset1.map(tokenize, batched=True, remove_columns=dataset1['train'].column_names)
tokenized_dataset

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
})

In [7]:
!pip install -U bitsandbytes>=0.46.1

In [6]:
from transformers import BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model

# Configure the 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# Load the model with quantization
model1 = AutoModelForSeq2SeqLM.from_pretrained(
    model1,
    quantization_config=bnb_config,
    device_map="auto",
)
model1

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear4bit(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear4bit(in_features=4096, out_features=1

In [7]:
# Configure LoRA (PEFT)
from peft import prepare_model_for_kbit_training

model1.gradient_checkpointing_enable()

model1 = prepare_model_for_kbit_training(model1)

peft_config = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.1,
    r=8,
    task_type=TaskType.SEQ_2_SEQ_LM,
)

model1 = get_peft_model(model1, peft_config)
model1.config.use_cache = False # We stop the cache during the training !
model1.print_trainable_parameters()

trainable params: 1,179,648 || all params: 407,470,080 || trainable%: 0.2895


In [9]:
trainer.state.log_history

[{'loss': 8.936598205566407,
  'grad_norm': 6.450990676879883,
  'learning_rate': 4.95114006514658e-05,
  'epoch': 0.010860711376595167,
  'step': 10},
 {'loss': 8.575479125976562,
  'grad_norm': 6.107723712921143,
  'learning_rate': 4.8968512486427796e-05,
  'epoch': 0.021721422753190334,
  'step': 20},
 {'loss': 7.794524383544922,
  'grad_norm': 4.650627136230469,
  'learning_rate': 4.84256243213898e-05,
  'epoch': 0.0325821341297855,
  'step': 30},
 {'loss': 7.883032989501953,
  'grad_norm': 6.7888054847717285,
  'learning_rate': 4.788273615635179e-05,
  'epoch': 0.04344284550638067,
  'step': 40},
 {'loss': 7.184727478027344,
  'grad_norm': 4.541581153869629,
  'learning_rate': 4.733984799131379e-05,
  'epoch': 0.054303556882975834,
  'step': 50}]

In [ ]:
# Now the training of the first model

model1.config.use_cache = False

data_collator = DataCollatorForSeq2Seq(tokenizer1, model=model1, label_pad_token_id=-100) # label_pad_token forces padding ignore in loss calculation

training_args = Seq2SeqTrainingArguments(
    output_dir = "./lora_bart_samsum_results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    eval_strategy='epoch',
    learning_rate=5e-5,
    num_train_epochs=1,
    weight_decay=0.01,
    fp16=True,
    logging_steps=1,
    logging_first_step=True,
    save_strategy='epoch',
    report_to='none', # We don't want wandb logging in this case
)

trainer = Seq2SeqTrainer(
    model=model1,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset['validation'],
    processing_class=tokenizer1,
)

trainer.train()

# We're going to implement a custom log loss function, because in my case the default hugging face one bugged out..
print("\nTraining loss history:\n")

for log in trainer.state.log_history:
  if 'loss' in log:
    print(
        f"Step: {log['step']:.4f}, Perplexity: {torch.exp(log['loss']):.4f}, Epoch: {log['loss']:.4f}"
    )

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss


In [ ]:
trainer.save_model("./best_lora_adapter")

Now as we're done with the first model we're going to continue with the second one, which is a roberta-base pretrained on the squad2 dataset

In [ ]:
from transformers import AutoModelForQuestionAnswering

model2_name = "deepset/roberta-base-squad2"

model2 = AutoModelForQuestionAnswering.from_pretrained(model2_name, device_map='auto')

tokenizer2 = AutoTokenizer.from_pretrained(model2)

In [ ]:
!pip install -q pypdf python-docx gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 19.7 MB/s eta 0:00:00


Now we're going to implement the gradio interface and the document parsing functions

In [ ]:
import gradio as gr
import os
from pypdf import PdfReader
from docx import Document
from transformers import pipeline

# Load the models using HF Pipelines
summarizer = pipeline("text2text-generation", model=model1, tokenizer=tokenizer1) # No device_map since the model is already inputed into the GPU via peft and bnb

qa = pipeline('question-answering', model=model2, tokenizer=tokenizer2, device=0)


def extract_text_from_file(file):
    if file is None:
        return ""

    file_path = file.name
    ext = os.path.splitext(file_path)[1].lower()
    extracted_text = ""

    try:
        if ext == ".pdf":
            reader = PdfReader(file_path)
            for page in reader.pages:
                text = page.extract_text()
                if text:
                    extracted_text += text + "\n"
        elif ext in [".docx", ".doc"]:
            doc = Document(file_path)
            for para in doc.paragraphs:
                text = para.text
                if text:
                    extracted_text += text + "\n"
        elif ext == ".txt":
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                extracted_text = f.read()
        else:
            return "Unsupported file format. Please upload PDF, DOCX, or TXT"

        return extracted_text.strip()

    except Exception as e:
        return f"Error reading file: {str(e)}"

def process_summary(text, file):
    source_text = extract_text_from_file(file) if file else text
    if not source_text.strip():
        return "Please enter text or upload a file."

    # Quick fix for ultra-long docs to avoid pipeline crash
    input_text = source_text[:4000]
    prompt = f"Summarize the following text short and clear:\n{input_text}\nSummary:"

    outputs = summarizer(prompt, max_new_tokens=150, do_sample=False)
    return outputs[0]["generated_text"].split("Summary:")[-1].strip()

def process_qa(text, file, question):
    source_text = extract_text_from_file(file) if file else text

    if not source_text:
        return "Please provide a document or text to answer the question."

    if not question:
        return "Please enter a question to answer."

    prompt = f"Context: {source_text}\nQuestion: {question}\nAnswer:"

    result = qa(prompt, max_new_tokens=30)
    return result[0]["generated_text"].split("Answer:")[-1].strip()



Loading weights:   0%|          | 0/316 [00:00<?, ?it/s]

[transformers] BartForCausalLM LOAD REPORT from: facebook/bart-large-cnn
Key                                                       | Status     |  | 
----------------------------------------------------------+------------+--+-
model.encoder.layers.{0...11}.fc1.bias                    | UNEXPECTED |  | 
model.encoder.layers.{0...11}.final_layer_norm.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn_layer_norm.bias   | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.out_proj.weight   | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc1.weight                  | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.out_proj.bias     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc2.weight                  | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.k_proj.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.final_layer_norm.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.v_proj.weight     | UNEXPECTED |  | 
mod

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] This checkpoint seem corrupted. The tied weights mapping for this model specifies to tie lm_head.bias to lm_head.decoder.bias, but both are absent from the checkpoint, and we could not find another related tied weight for those keys
[transformers] RobertaForCausalLM LOAD REPORT from: deepset/roberta-base-squad2
Key                       | Status     | 
--------------------------+------------+-
qa_outputs.bias           | UNEXPECTED | 
qa_outputs.weight         | UNEXPECTED | 
lm_head.decoder.bias      | MISSING    | 
lm_head.dense.weight      | MISSING    | 
lm_head.layer_norm.weight | MISSING    | 
lm_head.layer_norm.bias   | MISSING    | 
lm_head.bias              | MISSING    | 
lm_head.dense.bias        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Build the Gradio Interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
  gr.Markdown("# 📄 Multi-Format Document Assistant")
  gr.Markdown("AI-powered portofolio tool to summarize text and extract precise answers from documents.")

  with gr.Tab("Document Summarizer"):
    with gr.Row():
      with gr.Column():
        sum_text = gr.Textbox(label="Option A: Paste text directly", lines=6, placeholder="Enter text...")
        sum_file = gr.File(label="Option B: Upload Document (PDF, DOCX, TXT)", file_types=[".pdf", ".docx", ".txt"])
        summary_btn = gr.Button("Generate Summary", variant="primary")
      with gr.Column():
        summary_output = gr.Textbox(label='Summary Output', lines=6, interactive=False)

    summary_btn.click(fn=process_summary, inputs=[sum_text, sum_file], outputs=summary_output)

  with gr.Tab("Document QA System"):
    with gr.Row():
      with gr.Column():
        qa_text = gr.Textbox(label="Option A: Paste context directly", lines=6, placeholder="Enter context...")
        qa_file = gr.File(label="Option B: Upload Document (PDF, DOCX, TXT)", file_types=[".pdf", ".docx", ".txt"])
        qa_question = gr.Textbox(label="Your Question", lines=2, placeholder="What is the main revenue driver mentioned?")
        qa_btn = gr.Button("Find Answer", variant="primary")
      with gr.Column():
        qa_output = gr.Textbox(label='Extracted Answer', lines=4, interactive=False)

    qa_btn.click(fn=process_qa, inputs=[qa_text, qa_file, qa_question], outputs=qa_output)

# Launch the app with a public share link
demo.launch(share=True)


KeyboardInterrupt: 